# ROGII Prefix Backtest Trap: More Cuts Can Hurt

Multi-cut prefix validation is widely recommended for this competition, and the recommendation is directionally correct: a tracker must be tested without exposing future TVT. However, synthetic cuts can still be distribution-shifted. This notebook demonstrates a concrete failure mode on all 773 training wells.

We study the smallest possible model: extrapolate the direct TVT slope from the last 30 visible rows, then shrink that extrapolation toward the last-known-TVT anchor. A global coefficient learned at the supplied prediction boundary gives a small honest improvement. In contrast, per-well coefficients learned from earlier 40/60/80% prefix cuts become much too aggressive and worsen performance at the real boundary.

This is a validation notebook only. It does not create `submission.csv` and does not use leaderboard feedback.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data was not found')
TRAIN_DIR = DATA_ROOT / 'train'
PATHS = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
WINDOW = 30
EARLY_CUTS = (0.4, 0.6, 0.8)
print('Data root:', DATA_ROOT)
print('Training wells:', len(PATHS))

## One-dimensional shrinkage model

For a prediction boundary `b`, let `anchor = TVT[b-1]` and estimate a clipped linear slope `s` on the last 30 prefix rows. The raw adjustment for an evaluation row is `d = s * (MD - MD[b-1])`. Our prediction is `anchor + alpha*d`.

For fixed rows, squared error is quadratic in `alpha`. If `e = anchor - truth`, then the exact least-squares coefficient is `-sum(e*d)/sum(d^2)`, clipped to a safe non-negative interval. This lets us audit the validation design without an optimizer or a leaderboard.

In [ ]:
def supplied_boundary(values: pd.Series) -> int:
    observed = values.notna().to_numpy()
    missing = np.flatnonzero(~observed)
    if len(missing) == 0 or missing[0] < WINDOW + 2 or observed[missing[0]:].any():
        raise ValueError('Expected a long visible prefix and one missing suffix')
    return int(missing[0])

def tail_slope(md, tvt, boundary):
    x = md[max(0, boundary-WINDOW):boundary]
    y = tvt[max(0, boundary-WINDOW):boundary]
    xc = x - x.mean()
    denominator = float(xc @ xc)
    slope = 0.0 if denominator <= 1e-12 else float(xc @ (y-y.mean()) / denominator)
    return float(np.clip(slope, -0.1, 0.1))

def quadratic_stats(md, tvt, boundary, end):
    anchor = float(tvt[boundary-1])
    error = anchor - tvt[boundary:end]
    adjustment = tail_slope(md, tvt, boundary) * (md[boundary:end] - md[boundary-1])
    return float(adjustment@adjustment), float(error@adjustment), float(error@error), len(error)

def fitted_alpha(a, b, maximum=1.0):
    return 0.0 if a <= 1e-12 else float(np.clip(-b/a, 0.0, maximum))

def pooled_rmse(frame, alpha_column):
    value = frame.C + 2*frame[alpha_column]*frame.B + frame[alpha_column]**2*frame.A
    return math.sqrt(float(value.sum()) / int(frame.n.sum()))

In [ ]:
records = []
for path in PATHS:
    frame = pd.read_csv(path, usecols=['MD', 'TVT', 'TVT_input'])
    md = frame.MD.to_numpy(dtype=float)
    tvt = frame.TVT.to_numpy(dtype=float)
    boundary = supplied_boundary(frame.TVT_input)
    A, B, C, n = quadratic_stats(md, tvt, boundary, len(frame))

    calibration_A = calibration_B = calibration_C = 0.0
    for fraction in EARLY_CUTS:
        cut = int(round(boundary*fraction))
        cut = max(WINDOW+2, min(boundary-1, cut))
        a, b, c, _ = quadratic_stats(md, tvt, cut, boundary)
        calibration_A += a
        calibration_B += b
        calibration_C += c

    records.append({
        'well_id': path.name.removesuffix('__horizontal_well.csv'),
        'A': A, 'B': B, 'C': C, 'n': n,
        'cal_A': calibration_A, 'cal_B': calibration_B, 'cal_C': calibration_C,
    })
stats = pd.DataFrame(records)
stats['fold'] = np.arange(len(stats)) % 5
print('Evaluation rows:', int(stats.n.sum()))

## Honest supplied-boundary estimate

We first learn one global coefficient with five-fold grouping by well. Every validation well receives a coefficient fitted only on the other wells. The final deployment coefficient is then fit on all training wells.

In [ ]:
stats['alpha_anchor'] = 0.0
stats['alpha_oof'] = np.nan
fold_alphas = []
for fold in range(5):
    train_mask = stats.fold != fold
    alpha = fitted_alpha(stats.loc[train_mask, 'A'].sum(), stats.loc[train_mask, 'B'].sum())
    fold_alphas.append(alpha)
    stats.loc[~train_mask, 'alpha_oof'] = alpha
global_alpha = fitted_alpha(stats.A.sum(), stats.B.sum())
stats['alpha_full'] = global_alpha
true_start_results = pd.Series({
    'anchor pooled RMSE': pooled_rmse(stats, 'alpha_anchor'),
    'five-fold OOF trend pooled RMSE': pooled_rmse(stats, 'alpha_oof'),
    'full-fit trend pooled RMSE': pooled_rmse(stats, 'alpha_full'),
    'full-fit alpha': global_alpha,
})
display(true_start_results.to_frame('value').style.format('{:.6f}'))
print('Fold alphas:', [round(value, 6) for value in fold_alphas])

## The early-prefix trap

Now each well fits its own coefficient using only truth before the supplied boundary. This is target-safe in the literal sense: no real suffix label is exposed. Yet it can still fail because earlier lateral behavior is not distributed like the supplied prediction start.

We cap local coefficients at 0.10, four times the global coefficient. Even with that cap, most local fits saturate. We also test a conservative gate that either uses the small global coefficient or reverts to anchor depending on early-cut error.

In [ ]:
stats['alpha_local'] = [fitted_alpha(a, b, maximum=0.10) for a, b in zip(stats.cal_A, stats.cal_B)]
cal_anchor = stats.cal_C
cal_global = stats.cal_C + 2*global_alpha*stats.cal_B + global_alpha**2*stats.cal_A
stats['alpha_gate'] = np.where(cal_global < cal_anchor, global_alpha, 0.0)
comparison = pd.Series({
    'anchor': pooled_rmse(stats, 'alpha_anchor'),
    'global full-fit trend': pooled_rmse(stats, 'alpha_full'),
    'early-cut global gate': pooled_rmse(stats, 'alpha_gate'),
    'early-cut per-well alpha': pooled_rmse(stats, 'alpha_local'),
    'fraction gated on': float((stats.alpha_gate > 0).mean()),
    'fraction local alpha at cap': float((stats.alpha_local >= 0.10-1e-12).mean()),
})
display(comparison.to_frame('value').style.format('{:.6f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(stats.alpha_local, bins=25, color='#8b5cf6', alpha=0.85)
axes[0].axvline(global_alpha, color='black', linestyle='--', label='global true-start alpha')
axes[0].set(title='Early-cut local coefficients saturate', xlabel='local alpha', ylabel='wells')
axes[0].legend()
anchor_well = np.sqrt(stats.C/stats.n)
local_sse = stats.C + 2*stats.alpha_local*stats.B + stats.alpha_local**2*stats.A
local_well = np.sqrt(local_sse/stats.n)
axes[1].scatter(anchor_well, local_well-anchor_well, s=11, alpha=0.45)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set(title='Transfer to real boundary', xlabel='anchor well RMSE', ylabel='local trend minus anchor RMSE')
plt.tight_layout()
plt.show()

## Practical conclusions

1. **Keep the supplied boundary as a first-class benchmark.** It is the closest local replay of deployment available in every training well.
2. **Use multi-cut validation, but match the cut regime.** Early cuts can systematically overestimate trend persistence and tracker confidence.
3. **Separate tuning from reporting by well.** Even a one-parameter model should report out-of-fold performance, fold coefficient stability, pooled RMSE, and tail metrics.
4. **A prefix-only gate is necessary but not sufficient.** A gate can be free of target leakage and still be distribution-shifted.
5. **Do not spend submissions on locally resolved micro-changes.** The OOF trend gain here is small; the useful output is the validation lesson, not a new leaderboard point.